# Burgers $\beta \rightarrow \alpha$ Transformation in Zirconium

In 1934 W. G. Burgers worked out how the body-centred cubic
high-temperature phase of **zirconium** turns into the
hexagonal close-packed low-temperature phase, and in doing so
defined the orientation relationship that now carries his name.
Zirconium is therefore not merely *an* example of the Burgers
relationship — it is *the* original one.

This notebook rebuilds that analysis from first principles using
PyTex, and pushes it further than the Kurdjumov-Sachs treatment in
notebooks 18-21 in four specific ways:

1. **Every number is derived twice.** Each quantity PyTex computes
   is checked against an independent closed-form expression written
   out in the text. Nothing is asserted that is not also verified in
   a cell below it.
2. **The transformation strain is decomposed analytically.** All
   three principal strains of the Burgers distortion in Zr are given
   exact algebraic forms in $a_\beta$, $a_\alpha$ and $c_\alpha$,
   and matched to the numerically computed stretch tensor.
3. **The intervariant misorientation spectrum is linked to the
   mechanism.** The classic $10.53^\circ$ intervariant
   misorientation is shown to be exactly
   $\arccos(1/3) - 60^\circ$, i.e. the very shear that regularises
   the distorted hexagon on $\{110\}_\beta$.
4. **Zirconium behaves differently from titanium.** Because
   $c_\alpha/a_\alpha$ for Zr sits further from ideal, the
   $\{110\}_\beta / (0002)_\alpha$ diffraction coincidence that is
   essentially exact in Ti is *resolvably split* in Zr. That
   difference is computed, not asserted.

**Contents**

| § | Topic |
|---|-------|
| 1 | The two phases of zirconium |
| 2 | Why $\{110\}_\beta \rightarrow (0001)_\alpha$: the distorted hexagon |
| 3 | The orientation relationship and its transformation matrix |
| 4 | Variants: group theory, packets, and the full variant table |
| 5 | Intervariant misorientations and the $10.53^\circ$ signature |
| 6 | Lattice correspondence, the shuffle, and the transformation strain |
| 7 | Parent and product unit cells rendered in the Burgers orientation |
| 8 | Composite SAED patterns down several $\beta$ zone axes |
| 9 | Variant pole figures |

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

# The pinned alpha-Zr CIF stores its symmetry as a Hermann-Mauguin symbol
# rather than an explicit operator list, which pymatgen's parser notes twice
# on load. The fixture is hash-checked, so the notes carry no information
# here and would only clutter the rendered page.
warnings.filterwarnings("ignore", category=UserWarning, module="pymatgen.*")

from pytex import (
    AtomicSite,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalPlane,
    CrystalPlaneOverlay,
    FrameDomain,
    Handedness,
    Lattice,
    MillerIndex,
    Orientation,
    OrientationRelationship,
    Phase,
    PlacedCrystal,
    ReferenceFrame,
    Rotation,
    SymmetrySpec,
    Transform3D,
    UnitCell,
    WorldScene3D,
    ZoneAxis,
    build_crystal_scene,
    find_parallel_planes,
    get_phase_fixture,
    intervariant_misorientations,
    plot_variant_pole_figure,
    render_world_scene_3d,
    variant_close_packed_groups,
    variant_pole_figure,
)
from pytex.diffraction.composite import (
    find_spot_coincidences,
    simulate_composite_saed,
)
from pytex.diffraction.kinematic import KinematicSimulationConfig
from pytex.plotting.composite_saed import (
    CompositeSAEDPlotConfig,
    SpotAnnotationConfig,
    SpotStyle,
    render_composite_saed,
)

np.set_printoptions(precision=6, suppress=True)
plt.rcParams["figure.dpi"] = 110

## 1. The two phases of zirconium

Zirconium is allotropic. Below about 1136 K it is **hcp**
($\alpha$-Zr, space group $P6_3/mmc$); above it, **bcc**
($\beta$-Zr, space group $Im\bar{3}m$).

$\alpha$-Zr is taken from the repository's pinned CIF fixture, so
its lattice parameters are the audited, hash-checked values used
everywhere else in PyTex rather than numbers retyped into this
notebook. $\beta$-Zr is built explicitly because the high-temperature
phase is not retainable at room temperature and so is not part of the
room-temperature fixture corpus; $a_\beta = 3.574$ Å is the standard
value extrapolated from high-temperature diffraction.

A point worth stating early, because the whole transformation
crystallography depends on it: the axial ratio of $\alpha$-Zr is
**below** the ideal close-packed value.

$$\frac{c_\alpha}{a_\alpha} = 1.5925 < \sqrt{8/3} = 1.6330$$

That $2.5\%$ deficit is why zirconium's Burgers crystallography is
quantitatively different from titanium's, as §8 will show in the
diffraction pattern.

In [ ]:
A_BETA = 3.574  # beta-Zr bcc lattice parameter (Angstrom), high-T extrapolation

# --- alpha-Zr: from the pinned, hash-checked CIF fixture -------------------
alpha_frame = ReferenceFrame(
    "alpha_zr_crystal", FrameDomain.CRYSTAL, ("a", "b", "c"), Handedness.RIGHT
)
alpha_record = get_phase_fixture("zr_hcp")
alpha_zr = alpha_record.load_phase(crystal_frame=alpha_frame)

A_ALPHA = alpha_zr.lattice.a
C_ALPHA = alpha_zr.lattice.c

print(f"fixture id      : {alpha_record.fixture_id}")
print(f"source          : {alpha_record.metadata['source_family']} "
      f"#{alpha_record.metadata['source_record_id']}")
print(f"sha256 (cif)    : {alpha_record.artifact_sha256[:16]}...")
print(f"space group     : {alpha_record.metadata['space_group_symbol']} "
      f"(#{alpha_record.metadata['space_group_number']})")
print(f"a, c            : {A_ALPHA:.4f}, {C_ALPHA:.4f} A")
print(f"c/a             : {C_ALPHA / A_ALPHA:.6f}   (ideal {np.sqrt(8 / 3):.6f})")
print(f"deficit         : {100 * (C_ALPHA / A_ALPHA / np.sqrt(8 / 3) - 1):+.3f} %")

In [ ]:
# --- beta-Zr: constructed bcc, two atoms per conventional cell -------------
beta_frame = ReferenceFrame(
    "beta_zr_crystal", FrameDomain.CRYSTAL, ("a", "b", "c"), Handedness.RIGHT
)
beta_lattice = Lattice(
    A_BETA, A_BETA, A_BETA, 90.0, 90.0, 90.0, crystal_frame=beta_frame
)
beta_zr = Phase(
    "beta_zr",
    lattice=beta_lattice,
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=beta_frame),
    crystal_frame=beta_frame,
    unit_cell=UnitCell(
        lattice=beta_lattice,
        sites=(
            AtomicSite("Zr1", "Zr", np.array([0.0, 0.0, 0.0])),
            AtomicSite("Zr2", "Zr", np.array([0.5, 0.5, 0.5])),
        ),
    ),
    space_group_symbol="Im-3m",
    space_group_number=229,
    chemical_formula="Zr",
)

print(f"beta-Zr  a = {A_BETA:.4f} A, point group {beta_zr.symmetry.point_group}, "
      f"{len(beta_zr.symmetry.operators)} proper operators")
print(f"alpha-Zr        point group {alpha_zr.symmetry.point_group}, "
      f"{len(alpha_zr.symmetry.operators)} proper operators")

### 1.1 The transformation is nearly volume-conserving

Two atoms occupy the bcc conventional cell and two the hcp cell, so
the atomic volumes are

$$\Omega_\beta = \frac{a_\beta^{3}}{2}, \qquad
\Omega_\alpha = \frac{\sqrt{3}}{4}\,a_\alpha^{2} c_\alpha .$$

The resulting dilatation is small and *positive* — the close-packed
phase is the less dense one here, a direct consequence of the
sub-ideal axial ratio.

In [ ]:
omega_beta = A_BETA**3 / 2.0
omega_alpha = np.sqrt(3.0) / 4.0 * A_ALPHA**2 * C_ALPHA
volume_change = omega_alpha / omega_beta - 1.0

print(f"Omega_beta   = {omega_beta:8.4f} A^3 / atom")
print(f"Omega_alpha  = {omega_alpha:8.4f} A^3 / atom")
print(f"dV/V         = {100 * volume_change:+.4f} %")

# Nearest-neighbour distances: bcc <111>/2 vs hcp basal a.
nn_beta = A_BETA * np.sqrt(3.0) / 2.0
print(f"\nnearest neighbour  beta  a*sqrt(3)/2 = {nn_beta:.4f} A  (8 neighbours)")
print(f"nearest neighbour  alpha a           = {A_ALPHA:.4f} A  (6 in basal plane)")
print(f"                                       {100 * (A_ALPHA / nn_beta - 1):+.3f} %")

## 2. Why $\{110\}_\beta \rightarrow (0001)_\alpha$: the distorted hexagon

The Burgers relationship is not an arbitrary pairing of planes. It
is forced by the fact that the **$\{110\}$ plane of a bcc lattice is
already almost close-packed**, and needs only a small shear to become
exactly so.

Work in the $(110)_\beta$ plane. Two in-plane basis vectors are
$[00\bar{1}]$ and $[1\bar{1}0]$. Take an atom at the origin; which
lattice translations lie *in* that plane?

- $\pm\tfrac{1}{2}[1\bar{1}1]$ and $\pm\tfrac{1}{2}[1\bar{1}\bar{1}]$,
  each of length $a_\beta\sqrt{3}/2$ — these are the bcc
  nearest-neighbour $\langle 111 \rangle$ vectors;
- $\pm[001]$, of length $a_\beta$.

That is **six** in-plane neighbours: a hexagon. But not a regular
one. Setting up in-plane coordinates
$\mathbf{e}_1 = [001]$, $\mathbf{e}_2 = [1\bar{1}0]/\sqrt{2}$, the six
neighbours sit at

$$(\pm a_\beta,\, 0), \qquad
\left(\pm \tfrac{a_\beta}{2},\, \pm \tfrac{a_\beta}{\sqrt{2}}\right),$$

so four of them are at $a_\beta\sqrt{3}/2$ and two at $a_\beta$, and
the angles between successive neighbour bonds,
measured at the central atom, alternate

$$\theta_1 = \arccos\!\left(\tfrac{1}{3}\right) = 70.529^\circ,
\qquad
\theta_2 = \arccos\!\left(\tfrac{1}{\sqrt{3}}\right) = 54.736^\circ .$$

A regular hexagon — the hcp basal plane — needs all six at
$60^\circ$. **The Burgers shear is precisely the operation that
carries $70.529^\circ$ into $60^\circ$**, a change of

$$\Delta = \arccos(1/3) - 60^\circ = 10.529^\circ .$$

Hold on to that number: it reappears in §5 as the smallest
intervariant misorientation of the whole transformation, and that is
no coincidence.

In [ ]:
# The six in-plane neighbours of the (110)_beta mesh, computed from the
# lattice rather than typed in: enumerate short bcc lattice translations
# and keep those perpendicular to [110].
plane_normal = np.array([1.0, 1.0, 0.0]) / np.sqrt(2.0)
e1 = np.array([0.0, 0.0, 1.0])
e2 = np.array([1.0, -1.0, 0.0]) / np.sqrt(2.0)

translations = []
for i in (-1, 0, 1):
    for j in (-1, 0, 1):
        for k in (-1, 0, 1):
            for shift in (np.zeros(3), np.full(3, 0.5)):
                v = (np.array([i, j, k], dtype=float) + shift) * A_BETA
                if np.linalg.norm(v) < 1e-9:
                    continue
                if abs(float(v @ plane_normal)) > 1e-9:
                    continue
                translations.append(v)

translations = np.array(translations)
lengths = np.linalg.norm(translations, axis=1)
keep = translations[lengths <= A_BETA + 1e-6]
mesh = np.array([[float(v @ e1), float(v @ e2)] for v in keep])
order = np.argsort(np.arctan2(mesh[:, 1], mesh[:, 0]))
mesh = mesh[order]

print(f"in-plane neighbours found: {len(mesh)}")
for point in mesh:
    print(f"   ({point[0]:+7.4f}, {point[1]:+7.4f})   |r| = "
          f"{np.linalg.norm(point):.4f} A")

bond_angles = []
for index in range(len(mesh)):
    u = mesh[index]
    w = mesh[(index + 1) % len(mesh)]
    cosine = float(u @ w) / (np.linalg.norm(u) * np.linalg.norm(w))
    bond_angles.append(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))

print(f"\nbond angles at the central atom (deg): {np.round(bond_angles, 4)}")
print(f"sum                                  : {sum(bond_angles):.4f} deg")
print(f"\narccos(1/3)      = {np.degrees(np.arccos(1 / 3)):.6f} deg")
print(f"arccos(1/sqrt3)  = {np.degrees(np.arccos(1 / np.sqrt(3))):.6f} deg")
print(f"Burgers shear    = arccos(1/3) - 60 = "
      f"{np.degrees(np.arccos(1 / 3)) - 60.0:.6f} deg")

In [ ]:
fig, (ax_mesh, ax_cmp) = plt.subplots(1, 2, figsize=(11.0, 5.0))

# --- left: the real (110)_beta mesh with its two distinct angles ---
ax_mesh.add_patch(MplPolygon(mesh, closed=True, facecolor="#88ccee",
                             edgecolor="#332288", alpha=0.35, lw=2.0))
ax_mesh.scatter(*mesh.T, s=170, color="#332288", zorder=4)
ax_mesh.scatter([0], [0], s=210, color="#cc6677", zorder=5)
for point, angle in zip(mesh, bond_angles):
    ax_mesh.plot([0, point[0]], [0, point[1]], color="#332288",
                 lw=1.0, ls=":", zorder=2)
    ax_mesh.annotate(f"{np.linalg.norm(point):.3f}",
                     xy=point * 0.55, fontsize=7.5, color="#332288",
                     ha="center", bbox=dict(fc="white", ec="none", alpha=0.75))
for index, point in enumerate(mesh):
    nxt = mesh[(index + 1) % len(mesh)]
    mid = 0.5 * (point + nxt) * 0.72
    ax_mesh.annotate(f"{bond_angles[index]:.2f}$^\\circ$", xy=mid, fontsize=9.0,
                     color="#882255", ha="center", va="center",
                     fontweight="bold",
                     bbox=dict(fc="white", ec="#882255", lw=0.5, alpha=0.85))
ax_mesh.set_title("$(110)_\\beta$ mesh: a distorted hexagon\n"
                  "4 bonds at $a\\sqrt{3}/2$, 2 at $a$", fontsize=10)

# --- right: distorted vs regular hexagon, area-matched ---
# Shoelace area of the distorted hexagon, matched by a regular one so the
# comparison isolates shape change from area change.
shifted = np.roll(mesh, -1, axis=0)
mesh_area = 0.5 * abs(float(
    np.sum(mesh[:, 0] * shifted[:, 1] - shifted[:, 0] * mesh[:, 1])
))
radius = np.sqrt(2.0 * mesh_area / (3.0 * np.sqrt(3.0)))

# Anchor the regular hexagon on the mesh's own angular positions (circular
# mean of the per-vertex offsets from a perfect 60 deg ladder). Starting it
# at 0 deg instead would pair each vertex with the wrong target and draw
# correspondence arrows that cross the figure.
mesh_angle = np.arctan2(mesh[:, 1], mesh[:, 0])
ladder = np.deg2rad(np.arange(6) * 60.0)
offset_angle = np.angle(np.mean(np.exp(1j * (mesh_angle - ladder))))
regular_angle = offset_angle + ladder
regular = np.column_stack([radius * np.cos(regular_angle),
                           radius * np.sin(regular_angle)])
ax_cmp.add_patch(MplPolygon(mesh, closed=True, facecolor="#88ccee",
                            edgecolor="#332288", alpha=0.30, lw=2.0,
                            label="$(110)_\\beta$  (bcc)"))
ax_cmp.add_patch(MplPolygon(regular, closed=True, facecolor="#117733",
                            edgecolor="#117733", alpha=0.22, lw=2.0, ls="--",
                            label="$(0001)_\\alpha$  (ideal hcp)"))
ax_cmp.scatter(*mesh.T, s=110, color="#332288", zorder=4)
ax_cmp.scatter(*regular.T, s=110, color="#117733", marker="D", zorder=4)
for point, target in zip(mesh, regular):
    ax_cmp.annotate("", xy=target, xytext=point,
                    arrowprops=dict(arrowstyle="->", color="#882255", lw=1.3))
ax_cmp.legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.14),
              ncol=2, frameon=False)
ax_cmp.set_title("The Burgers shear regularises the hexagon\n"
                 "$70.53^\\circ \\rightarrow 60^\\circ$ "
                 "($\\Delta = 10.53^\\circ$)", fontsize=10)

for axis in (ax_mesh, ax_cmp):
    axis.set_aspect("equal")
    axis.set_xlabel("$[001]_\\beta$  (A)")
    axis.set_ylabel("$[1\\bar{1}0]_\\beta$  (A)")
    axis.grid(alpha=0.2)
    axis.autoscale_view()
fig.tight_layout()

### 2.1 The three ingredients of the mechanism

The full $\beta \rightarrow \alpha$ transformation is
*reconstructive*, and Burgers decomposed it into three parts that
act together:

1. **A shear** on $\{112\}_\beta$ along
   $\langle 11\bar{1} \rangle_\beta$ of magnitude $\approx 0.22$,
   which opens the $70.53^\circ$ angle out to $60^\circ$ and turns
   the distorted hexagon into a regular one.
2. **A shuffle**: alternate $(110)_\beta$ planes must slide by
   $\approx \tfrac{1}{12}\langle 1\bar{1}0 \rangle$ so the stacking
   changes from the bcc $\ldots ABAB \ldots$ of $\{110\}$ to the true
   hcp $\ldots ABAB \ldots$ with the correct interlayer registry.
   This is a *non-affine* motion — no homogeneous strain can produce
   it — and §6 shows PyTex detecting exactly this through a
   correspondence matrix that is half-integer rather than integer.
3. **A small dilatation** adjusting the interplanar spacing
   $d_{110}^{\beta} \rightarrow d_{0002}^{\alpha}$.

Because step 2 exists, the Burgers transformation is *not* a pure
lattice deformation, which is precisely what distinguishes it from
the displacive Bain/KS family treated in notebook 19.

## 3. The orientation relationship and its transformation matrix

With the mechanism understood, the relationship itself is the pair of
parallelisms

$$\{110\}_\beta \parallel (0001)_\alpha, \qquad
\langle \bar{1}11 \rangle_\beta \parallel
\langle 11\bar{2}0 \rangle_\alpha .$$

The first says the pseudo-hexagonal plane becomes the basal plane;
the second says the bcc close-packed direction becomes the hcp
close-packed direction. Together they fix a rotation completely.

In [ ]:
burgers = OrientationRelationship.from_burgers_correspondence(
    parent_phase=beta_zr, child_phase=alpha_zr
)
print(burgers.describe())

In [ ]:
rotation_matrix = burgers.parent_to_child_rotation.as_matrix()

print("R  (parent Cartesian -> child Cartesian):")
print(rotation_matrix)
print(f"\ndet R                = {np.linalg.det(rotation_matrix):.12f}")
print(f"|R^T R - I|_max      = "
      f"{np.abs(rotation_matrix.T @ rotation_matrix - np.eye(3)).max():.3e}")
print(f"axis                 = {burgers.parent_to_child_rotation.axis}")
print(f"angle                = {burgers.parent_to_child_rotation.angle_deg:.6f} deg")
print(f"symmetry-reduced     = {burgers.misorientation().angle_deg:.6f} deg")

### 3.1 Verifying the parallelisms

An orientation relationship is only meaningful if it actually does
what it claims. Both defining parallelisms are checked here by mapping
the parent objects through $R$ and taking dot products with the child
objects — the answers must be $1$ to machine precision.

In [ ]:
parent_plane = CrystalPlane(MillerIndex(np.array([1, 1, 0]), phase=beta_zr),
                            phase=beta_zr)
child_plane = CrystalPlane.from_miller_bravais((0, 0, 0, 1), phase=alpha_zr)
parent_direction = CrystalDirection(np.array([-1.0, 1.0, 1.0]), phase=beta_zr)
child_direction = CrystalDirection.from_miller_bravais((1, 1, -2, 0),
                                                       phase=alpha_zr)

mapped_normal = rotation_matrix @ parent_plane.normal
mapped_direction = rotation_matrix @ parent_direction.unit_vector

print("plane parallelism   (110)_beta || (0001)_alpha")
print(f"   R n_beta        = {mapped_normal}")
print(f"   n_alpha         = {child_plane.normal}")
print(f"   dot             = {float(mapped_normal @ child_plane.normal):.15f}")
print("\ndirection parallelism   [-111]_beta || [11-20]_alpha")
print(f"   R d_beta        = {mapped_direction}")
print(f"   d_alpha         = {child_direction.unit_vector}")
print(f"   dot             = "
      f"{float(mapped_direction @ child_direction.unit_vector):.15f}")

# The two objects must also be mutually perpendicular in each phase.
print(f"\nn.d in beta        = "
      f"{float(parent_plane.normal @ parent_direction.unit_vector):.3e}")
print(f"n.d in alpha       = "
      f"{float(child_plane.normal @ child_direction.unit_vector):.3e}")

## 4. Variants: group theory, packets, and the variant table

### 4.1 Why exactly twelve

The variant count is a group-theoretic statement, not an empirical
one. A single parent orientation can produce
$|G_\beta| / |G_\beta \cap G_\alpha^{\,\text{OR}}|$ distinct child
orientations, where $G_\beta$ is the proper point group of the parent
($432$, order 24) and the intersection group contains the parent
operations that leave the child lattice invariant.

For Burgers that intersection has order 2 — the identity plus the
two-fold about the $\langle \bar{1}11 \rangle$ axis shared with the
child — giving

$$N = \frac{24}{2} = 12 .$$

In [ ]:
variants = burgers.generate_variants()
raw_variants = burgers.generate_variants(reduce_by_child_symmetry=False)

parent_order = len(beta_zr.symmetry.operators)
child_order = len(alpha_zr.symmetry.operators)

print(f"|G_beta| (proper)               = {parent_order}")
print(f"|G_alpha| (proper)              = {child_order}")
print(f"distinct variants               = {len(variants)}")
print(f"|G_beta| / n_variants           = {parent_order // len(variants)}"
      f"   <- order of the intersection group")
print(f"raw operator products           = {len(raw_variants)}"
      f"   (= {parent_order} x {child_order}, every symmetry-equivalent "
      f"description counted)")
assert len(variants) == 12

### 4.2 Packets

Each variant carries exactly one member of the
$\{110\}_\beta$ family onto the basal plane. Variants sharing that
member form a **packet**. The $\{110\}$ family has six
crystallographically distinct members (12 planes, antipodally
collapsed), so the twelve Burgers variants split into
**six packets of two** — the structural unit of the basketweave
and colony morphologies seen in Zr and Ti microstructures.

In [ ]:
packets = variant_close_packed_groups(burgers, parent_plane)
print(f"packet label per variant : {packets}")
print(f"number of packets        : {len(set(packets.tolist()))}")
for packet_id in sorted(set(packets.tolist())):
    members = [i + 1 for i, p in enumerate(packets) if p == packet_id]
    print(f"   packet {packet_id}: variants {members}")

In [ ]:
# Which {110}_beta member and which child plane each variant pairs.
report = find_parallel_planes(burgers, parent_plane, tolerance_deg=0.5)
print(report.describe()[:700])

In [ ]:
# Full variant table: axis/angle of each variant rotation, plus its packet.
print(f"{'V':>3}  {'angle (deg)':>11}  {'axis (parent Cartesian)':<34} packet")
print("-" * 66)
for variant in variants:
    rotation = variant.parent_to_child_rotation
    axis = rotation.axis
    print(f"{variant.variant_index:>3}  {rotation.angle_deg:>11.4f}  "
          f"[{axis[0]:+.4f} {axis[1]:+.4f} {axis[2]:+.4f}]   "
          f"{packets[variant.variant_index - 1]}")

## 5. Intervariant misorientations and the $10.53^\circ$ signature

Twelve variants give $\binom{12}{2} = 66$ unordered pairs. Reduced by
the hexagonal child symmetry, those 66 disorientations collapse onto
only **five** distinct values. This spectrum is the standard
experimental fingerprint of the Burgers relationship: it is what an
EBSD misorientation-angle histogram of a transformed $\beta$ grain
should look like, and it is how the relationship is confirmed in
practice.

The literature values (Gey & Humbert; Wang, Aindow & Starink) are
$10.53^\circ$, $60.00^\circ$, $60.83^\circ$, $63.26^\circ$ and
$90.00^\circ$. PyTex computes them below from nothing but the two
point groups and the defining rotation.

In [ ]:
pairs = intervariant_misorientations(burgers)
angles = np.array([pair.angle_deg for pair in pairs])
print(f"variant pairs: {len(pairs)}  (= C(12,2) = {12 * 11 // 2})")

unique_angles, counts = np.unique(np.round(angles, 4), return_counts=True)
literature = {10.53: "10.53 <0001>", 60.0: "60.00 <11-20>",
              60.83: "60.83 <1.377 1 2.377 0.359>",
              63.26: "63.26 <10 5 5 3>", 90.0: "90.00 <1 2.38 1.38 0>"}

print(f"\n{'angle (deg)':>12} {'count':>6}  {'axis (child Cartesian)':<30} literature")
print("-" * 88)
for value, count in zip(unique_angles, counts):
    index = int(np.argmin(np.abs(angles - value)))
    axis = pairs[index].axis_child_frame
    key = min(literature, key=lambda k: abs(k - value))
    tag = literature[key] if abs(key - value) < 0.02 else "-"
    print(f"{value:>12.4f} {count:>6}  "
          f"[{axis[0]:+.4f} {axis[1]:+.4f} {axis[2]:+.4f}]   {tag}")
print(f"\ntotal accounted for: {counts.sum()}")

In [ ]:
fig, (ax_hist, ax_cum) = plt.subplots(1, 2, figsize=(11.0, 3.9))

palette = ["#cc6677", "#332288", "#ddcc77", "#117733", "#88ccee"]
ax_hist.bar(unique_angles, counts, width=1.6,
            color=palette[: len(unique_angles)], edgecolor="#222222", lw=0.7)
for value, count in zip(unique_angles, counts):
    ax_hist.annotate(f"{value:.2f}$^\\circ$\n$\\times${count}",
                     xy=(value, count), xytext=(0, 4),
                     textcoords="offset points", ha="center", fontsize=8)
ax_hist.set_xlabel("disorientation angle (deg)")
ax_hist.set_ylabel("number of variant pairs")
ax_hist.set_title("Burgers intervariant spectrum (66 pairs, 5 values)",
                  fontsize=10)
ax_hist.set_ylim(0, counts.max() * 1.28)
ax_hist.grid(alpha=0.2, axis="y")

ordered = np.sort(angles)
ax_cum.step(ordered, np.arange(1, len(ordered) + 1) / len(ordered),
            where="post", color="#332288", lw=2.0)
ax_cum.axvline(np.degrees(np.arccos(1 / 3)) - 60.0, color="#cc6677",
               ls="--", lw=1.4,
               label="$\\arccos(1/3)-60^\\circ = 10.53^\\circ$")
ax_cum.set_xlabel("disorientation angle (deg)")
ax_cum.set_ylabel("cumulative fraction")
ax_cum.set_title("Cumulative distribution", fontsize=10)
ax_cum.legend(fontsize=8.5, loc="lower right")
ax_cum.grid(alpha=0.2)
fig.tight_layout()

### 5.1 The $10.53^\circ$ pair is the hexagon shear

The smallest intervariant misorientation is $10.53^\circ$ about
$[0001]_\alpha$, and it occurs for exactly the six pairs of variants
that share a $\{110\}_\beta$ plane but use *different*
$\langle \bar{1}11 \rangle_\beta$ directions within it.

Its value is not an accident of the lattice parameters. Two
$\langle 111 \rangle$ directions lying in the same $\{110\}$ plane
are separated by $\arccos(1/3) = 70.53^\circ$; after both variants
have mapped their own direction onto a basal
$\langle 11\bar{2}0 \rangle$ (which are $60^\circ$ apart), the
residual rotation about the common basal normal is the difference:

$$10.53^\circ = \arccos\!\left(\tfrac{1}{3}\right) - 60^\circ .$$

The same $70.53^\circ$ that made the $(110)_\beta$ hexagon irregular
in §2 shows up here as the smallest angle in the misorientation
spectrum. It is a purely geometric constant of the bcc lattice, and it
carries no dependence on $a_\beta$, $a_\alpha$ or $c_\alpha$
whatsoever.

In [ ]:
angle_111 = np.degrees(np.arccos(1.0 / 3.0))
predicted = angle_111 - 60.0
observed = float(unique_angles[0])

print(f"angle between <111> pair in a common {{110}} : {angle_111:.6f} deg")
print(f"basal <11-20> separation                    : 60.000000 deg")
print(f"predicted smallest intervariant             : {predicted:.6f} deg")
print(f"computed  smallest intervariant             : {observed:.6f} deg")
print(f"difference                                  : "
      f"{abs(predicted - observed):.3e} deg")

# And the pairs carrying it are exactly the six same-packet pairs.
same_packet = [
    pair for pair in pairs
    if packets[pair.variant_a - 1] == packets[pair.variant_b - 1]
]
print(f"\nsame-packet pairs                : {len(same_packet)}")
print(f"their angles                     : "
      f"{np.unique(np.round([p.angle_deg for p in same_packet], 4))}")

## 6. Lattice correspondence, the shuffle, and the transformation strain

### 6.1 A half-integer correspondence exposes the shuffle

A lattice correspondence records where the parent basis vectors land
in child-lattice coordinates. For a purely displacive transformation
that matrix is **integer**: every parent lattice point becomes a child
lattice point.

For $\beta \rightarrow \alpha$ it cannot be. The bcc primitive cell
holds one atom and the hcp cell holds two, so two bcc lattice points
must map onto one hcp lattice point *plus one motif atom*. The motif
atom is not a lattice translation, and the discrepancy is the
**shuffle** of §2.1.

PyTex searches over bounded denominators and reports which one it
needed. A denominator of 2 is the signature of a reconstructive
transformation carrying half a cell by shuffle.

In [ ]:
strain_report = burgers.deformation_gradient()
print(strain_report.describe())

In [ ]:
print(f"correspondence denominator : {strain_report.correspondence_denominator}"
      f"   <- 2 means a shuffle carries half a cell")
print(f"max component error        : "
      f"{strain_report.correspondence_max_component_error:.4f}")
print("\nlattice correspondence (child-basis images of the parent basis):")
print(strain_report.correspondence)
print("\ndeformation gradient F (parent frame, rigid rotation removed):")
print(strain_report.deformation_gradient)
print("\nright stretch tensor U:")
print(strain_report.stretch_tensor)
print(f"\ndet F      = {np.linalg.det(strain_report.deformation_gradient):.8f}")
print(f"volume     = {strain_report.volume_ratio:.8f}  "
      f"({100 * (strain_report.volume_ratio - 1):+.4f} %)")
print(f"polar rot. = {strain_report.polar_rotation_deg:.6f} deg")

### 6.2 All three principal strains in closed form

This is the quantitative heart of the notebook. The Burgers
distortion acts on three mutually perpendicular directions, and each
has an exact algebraic image:

| parent direction | length before | length after | principal strain |
|---|---|---|---|
| $[1\bar{1}0]_\beta$ | $a_\beta\sqrt{2}$ | $a_\alpha\sqrt{3}$ | $\dfrac{a_\alpha\sqrt{3}}{a_\beta\sqrt{2}} - 1$ |
| $[110]_\beta$ (plane normal) | $a_\beta/\sqrt{2}$ | $c_\alpha/2$ | $\dfrac{c_\alpha}{a_\beta\sqrt{2}} - 1$ |
| $[001]_\beta$ | $a_\beta$ | $a_\alpha$ | $\dfrac{a_\alpha}{a_\beta} - 1$ |

The middle row is the dilatation of §2.1 item 3; the outer rows are
the in-plane regularisation of the hexagon. Their product must
reproduce the volume change computed in §1.1 — an independent check
that closes the loop.

In [ ]:
eps_1 = A_ALPHA * np.sqrt(3.0) / (A_BETA * np.sqrt(2.0)) - 1.0  # [1-10]_beta
eps_2 = C_ALPHA / (A_BETA * np.sqrt(2.0)) - 1.0                 # [110]_beta normal
eps_3 = A_ALPHA / A_BETA - 1.0                                  # [001]_beta

analytic = np.sort(np.array([eps_1, eps_2, eps_3]))
computed = np.sort(np.array(strain_report.principal_stretches) - 1.0)

print(f"{'source':<12} {'strain 1':>11} {'strain 2':>11} {'strain 3':>11}")
print("-" * 48)
print(f"{'analytic':<12} " + " ".join(f"{100 * v:>10.4f}%" for v in analytic))
print(f"{'PyTex':<12} " + " ".join(f"{100 * v:>10.4f}%" for v in computed))
print(f"{'difference':<12} "
      + " ".join(f"{100 * abs(a - c):>10.2e} " for a, c in zip(analytic, computed)))

print(f"\n[1-10]_beta : a_b*sqrt2 = {A_BETA * np.sqrt(2):.4f} A"
      f"  ->  a_a*sqrt3 = {A_ALPHA * np.sqrt(3):.4f} A   {100 * eps_1:+.4f} %")
print(f"[110]_beta  : d_110     = {A_BETA / np.sqrt(2):.4f} A"
      f"  ->  c_a/2     = {C_ALPHA / 2:.4f} A   {100 * eps_2:+.4f} %")
print(f"[001]_beta  : a_b       = {A_BETA:.4f} A"
      f"  ->  a_a       = {A_ALPHA:.4f} A   {100 * eps_3:+.4f} %")

product = (1 + eps_1) * (1 + eps_2) * (1 + eps_3)
print(f"\nproduct of stretches   = {product:.8f}  ({100 * (product - 1):+.4f} %)")
print(f"volume ratio (Sec 1.1) = {1 + volume_change:.8f}  "
      f"({100 * volume_change:+.4f} %)")
print(f"agreement              = {abs(product - 1 - volume_change):.3e}")

### 6.3 The residual rotation is the other hexagon angle

`polar_rotation_deg` reports how far the variant's rigid rotation sits
from the pure correspondence distortion. For Burgers in Zr it is
$5.26^\circ$ — and once again this is a bcc lattice constant in
disguise:

$$5.264^\circ = 60^\circ - \arccos\!\left(\tfrac{1}{\sqrt{3}}\right)
= 60^\circ - 54.736^\circ,$$

the *other* bond angle of the distorted hexagon of §2, measured
against the regular $60^\circ$. The two hexagon angles
($70.53^\circ$ and $54.74^\circ$) thus account for both the smallest
intervariant misorientation and the residual polar rotation.

In [ ]:
predicted_polar = 60.0 - np.degrees(np.arccos(1.0 / np.sqrt(3.0)))
print(f"60 - arccos(1/sqrt3)     = {predicted_polar:.6f} deg")
print(f"PyTex polar_rotation_deg = {strain_report.polar_rotation_deg:.6f} deg")
print(f"difference               = "
      f"{abs(predicted_polar - strain_report.polar_rotation_deg):.3e} deg")

print(f"\nhexagon angle 1  {np.degrees(np.arccos(1 / 3)):.4f} deg"
      f"  -> smallest intervariant misorientation "
      f"{np.degrees(np.arccos(1 / 3)) - 60:.4f} deg")
print(f"hexagon angle 2  {np.degrees(np.arccos(1 / np.sqrt(3))):.4f} deg"
      f"  -> residual polar rotation           {predicted_polar:.4f} deg")

## 7. Parent and product unit cells in the Burgers orientation

The placement convention matters and is easy to get backwards.
`parent_to_child_rotation` is the matrix $R$ that re-expresses a
*parent-frame* Cartesian vector in the *child* frame. To draw both
crystals in one world frame anchored on the parent, the child geometry
must therefore be placed by the **transpose**:

$$\mathbf{x}_\text{world} = R^{\mathsf{T}}\, \mathbf{x}_\text{child}.$$

The cell below asserts that convention before drawing anything: with
$R^{\mathsf{T}}$ applied, the child basal normal must coincide with
the parent $(110)$ normal and the child
$[11\bar{2}0]$ with the parent $[\bar{1}11]$.

In [ ]:
placement = rotation_matrix.T

basal_in_world = placement @ child_plane.normal
close_packed_in_world = placement @ child_direction.unit_vector

print("placement check (child geometry expressed in the parent world frame)")
print(f"   R^T n_(0001)   = {basal_in_world}")
print(f"   n_(110)_beta   = {parent_plane.normal}")
print(f"   dot            = {float(basal_in_world @ parent_plane.normal):.15f}")
print(f"\n   R^T d_[11-20]  = {close_packed_in_world}")
print(f"   d_[-111]_beta  = {parent_direction.unit_vector}")
print(f"   dot            = "
      f"{float(close_packed_in_world @ parent_direction.unit_vector):.15f}")

In [ ]:
beta_scene = build_crystal_scene(
    beta_zr,
    repeats=(1, 1, 1),
    render_style="ball_and_stick",
    plane_overlays=(
        CrystalPlaneOverlay(plane=parent_plane, label="$(110)_\\beta$",
                            color="#3182bd", alpha=0.55),
    ),
    direction_overlays=(
        CrystalDirectionOverlay(direction=parent_direction,
                                anchor_fractional=np.array([1.0, 0.0, 0.0]),
                                label="$[\\bar{1}11]_\\beta$", color="#cc6677"),
    ),
)

alpha_scene = build_crystal_scene(
    alpha_zr,
    repeats=(2, 2, 1),
    render_style="ball_and_stick",
    plane_overlays=(
        CrystalPlaneOverlay(plane=child_plane, label="$(0001)_\\alpha$",
                            color="#117733", alpha=0.55),
    ),
    direction_overlays=(
        CrystalDirectionOverlay(direction=child_direction,
                                anchor_fractional=np.array([0.0, 0.0, 0.0]),
                                label="$[11\\bar{2}0]_\\alpha$", color="#cc6677"),
    ),
)


def fill_3d_axes(axis, zoom=1.0):
    # Undistorted geometry that actually fills a 3D panel. Forcing the three
    # limits into a cube keeps proportions honest but wastes most of the
    # panel; taking the box aspect from the true data spans keeps proportions
    # honest *and* uses the space.
    limits = np.array([axis.get_xlim3d(), axis.get_ylim3d(), axis.get_zlim3d()])
    spans = limits[:, 1] - limits[:, 0]
    axis.set_box_aspect(tuple(spans / spans.max()), zoom=zoom)
    axis.set_axis_off()


fig = plt.figure(figsize=(11.0, 5.0))

ax_beta = fig.add_subplot(1, 2, 1, projection="3d")
render_world_scene_3d(
    WorldScene3D(crystals=(PlacedCrystal(beta_scene, Transform3D()),)),
    ax=ax_beta, elev_deg=18.0, azim_deg=28.0,
    title="$\\beta$-Zr (bcc, $Im\\bar{3}m$)\n"
          "$(110)$ shaded, $[\\bar{1}11]$ arrowed",
)
fill_3d_axes(ax_beta, zoom=1.25)

ax_alpha = fig.add_subplot(1, 2, 2, projection="3d")
render_world_scene_3d(
    WorldScene3D(crystals=(PlacedCrystal(alpha_scene, Transform3D()),)),
    ax=ax_alpha, elev_deg=18.0, azim_deg=28.0,
    title="$\\alpha$-Zr (hcp, $P6_3/mmc$)\n"
          "$(0001)$ shaded, $[11\\bar{2}0]$ arrowed",
)
fill_3d_axes(ax_alpha, zoom=1.25)
fig.tight_layout()

In [ ]:
# Offsetting the child along the SHARED normal, and viewing nearly along the
# common plane, makes the parallelism itself the visual message: both shaded
# facets appear as parallel slabs and both red arrows point the same way.
fig = plt.figure(figsize=(8.6, 5.8))
ax_or = fig.add_subplot(111, projection="3d")
render_world_scene_3d(
    WorldScene3D(crystals=(
        PlacedCrystal(beta_scene, Transform3D()),
        PlacedCrystal(
            alpha_scene,
            Transform3D(matrix=placement,
                        translation=parent_plane.normal * 7.5),
        ),
    )),
    ax=ax_or, elev_deg=16.0, azim_deg=-35.0, show_legend=False,
    title="Burgers OR placement: child rotated by $R^{\\mathsf{T}}$, offset "
          "along the shared normal\n"
          "$(110)_\\beta \\parallel (0001)_\\alpha$   and   "
          "$[\\bar{1}11]_\\beta \\parallel [11\\bar{2}0]_\\alpha$",
)
fill_3d_axes(ax_or, zoom=1.5)
fig.tight_layout()

That last figure is the geometric statement of the relationship: the
shaded $(110)_\beta$ facet and the shaded $(0001)_\alpha$ facet are
parallel slabs, and the red arrows — $[\bar{1}11]_\beta$ and
$[11\bar{2}0]_\alpha$ — point the same way. The two crystals are
translated apart *along their common normal* so both remain visible;
no relative rotation beyond $R^{\mathsf{T}}$ has been applied, which is
exactly what the two dot products above certify.

## 8. Composite SAED patterns down several $\beta$ zone axes

A selected-area aperture spanning a $\beta/\alpha$ interface records
the parent reflections together with those of every illuminated
variant. Which variants give *interpretable* (low-index) child
patterns depends entirely on which parent zone axis the beam is along.

### 8.1 A zone-axis survey

Two parent zone axes are special because they appear in the defining
parallelisms themselves:

- down $\langle 110 \rangle_\beta$ the beam is along the basal normal
  of the variants built on that plane, so those variants show the
  $[0001]_\alpha$ **basal** pattern;
- down $\langle 111 \rangle_\beta$ the beam is along a
  $\langle 11\bar{2}0 \rangle_\alpha$ direction.

$\langle 100 \rangle_\beta$ appears in *neither*, and the survey shows
the consequence directly: no variant lands on a rational child zone at
all.

In [ ]:
survey_axes = ([1, 1, 0], [1, 1, 1], [0, 0, 1], [1, 1, 2])
survey = {}

print(f"{'beam':<10} {'exact':>6}  {'max dev':>9}   child zones reached")
print("-" * 74)
for beam in survey_axes:
    composite = simulate_composite_saed(
        burgers, ZoneAxis(np.array(beam), phase=beta_zr)
    )
    survey[tuple(beam)] = composite
    deviations = np.array(
        [p.nearest_zone_axis.deviation_deg for p in composite.variant_patterns]
    )
    exact = [p for p in composite.variant_patterns
             if p.nearest_zone_axis.deviation_deg < 1e-6]
    labels = sorted({p.nearest_zone_axis.label() for p in exact}) or ["(none)"]
    print(f"{str(beam):<10} {len(exact):>6}  {deviations.max():>8.3f} deg   "
          f"{', '.join(labels)}")

That table is worth reading carefully.

- **$[110]_\beta$** — 2 of the 12 variants sit exactly on
  $[0001]_\alpha$. The remaining ten are within $1.32^\circ$ of a
  rational zone, which is small enough that they still produce
  recognisable, indexable patterns.
- **$[111]_\beta$** — 3 variants land exactly on
  $\langle 11\bar{2}0 \rangle_\alpha$.
- **$[001]_\beta$** — *no* variant reaches a rational child zone; the
  best is $2.78^\circ$ away. A composite pattern taken here is
  genuinely hard to index, which is exactly why the practical
  literature recommends tilting to $\langle 110 \rangle_\beta$ before
  attempting Burgers variant analysis.

### 8.2 The basal view down $[110]_\beta$

In [ ]:
composite_110 = survey[(1, 1, 0)]
basal_pattern = next(
    p for p in composite_110.variant_patterns
    if p.nearest_zone_axis.deviation_deg < 1e-6
)

print(f"basal variant      : {basal_pattern.label()}")
print(f"reflections        : {len(basal_pattern.spots)}")
print(f"all with l = 0     : "
      f"{bool(np.all(basal_pattern.spots.hkl[:, 2] == 0))}")
print(f"largest d-spacing  : "
      f"{basal_pattern.spots.d_spacing_angstrom.max():.6f} A")
print(f"analytic a*sqrt3/2 : {A_ALPHA * np.sqrt(3) / 2:.6f} A"
      f"   <- the {{10-10}} prism spacing")

# Six-fold symmetry, verified directly on the simulated coordinates.
coordinates = basal_pattern.spots.detector_mm
theta = np.deg2rad(60.0)
spin = np.array([[np.cos(theta), -np.sin(theta)],
                 [np.sin(theta), np.cos(theta)]])
worst = max(
    float(np.min(np.linalg.norm(coordinates - point, axis=1)))
    for point in coordinates @ spin.T
)
print(f"\nworst mismatch under a 60 deg rotation: {worst:.3e} mm"
      f"   <- the pattern is exactly six-fold")

In [ ]:
fig = render_composite_saed(
    composite_110.select_variants([1, 2]),
    config=CompositeSAEDPlotConfig(
        annotation=SpotAnnotationConfig(coincidence_tolerance_mm=2.0,
                                        max_labels=22),
        title=("Burgers composite SAED, Zr:  $\\beta$ $[110]$ "
               "+ basal $\\alpha$ variants"),
        figsize=(7.6, 6.4),
    ),
)

The hexagonal child automatically switches to four-index
Miller-Bravais labels ($(1\,1\,\bar{2}\,0)$, $(3\,0\,\bar{3}\,0)$,
...) while the cubic parent keeps three, so the two phases are
distinguishable by notation alone.

### 8.3 The zirconium fingerprint — and how it differs from titanium

The plane parallelism forces $\{110\}_\beta$ and $(0002)_\alpha$
reflections onto nearly the same detector radius. Their residual
separation at camera constant $L\lambda$ is

$$\Delta r = \left(\frac{\sqrt{2}}{a_\beta}
- \frac{2}{c_\alpha}\right) L\lambda .$$

In **titanium** ($a_\beta = 3.3065$, $c_\alpha = 4.6855$ Å) this
evaluates to $0.155$ mm — far inside a spot diameter, so the two
reflections are simply not resolvable and the composite reads as one
decorated pattern.

In **zirconium** the sub-ideal axial ratio pushes $c_\alpha/2$ away
from $a_\beta/\sqrt{2}$ by $1.83\%$, and the separation grows by
roughly a factor of eight. It becomes a *measurable split* — a
genuinely different experimental signature from the titanium case that
the textbook description is usually written around.

In [ ]:
CAMERA_CONSTANT = 180.0  # mm.Angstrom, the KinematicSimulationConfig default

d_110_beta = A_BETA / np.sqrt(2.0)
d_0002_alpha = C_ALPHA / 2.0
analytic_split = (np.sqrt(2.0) / A_BETA - 2.0 / C_ALPHA) * CAMERA_CONSTANT

print(f"d(110)_beta   = {d_110_beta:.6f} A")
print(f"d(0002)_alpha = {d_0002_alpha:.6f} A")
print(f"mismatch      = "
      f"{100 * (d_0002_alpha / d_110_beta - 1):+.4f} %")
print(f"analytic split at L.lambda = {CAMERA_CONSTANT:.0f} mm.A : "
      f"{abs(analytic_split):.6f} mm")

coincidences = find_spot_coincidences(composite_110, tolerance_mm=2.0)
match = next(
    c for c in coincidences.coincidences
    if tuple(np.abs(c.parent_hkl)) == (1, 1, 0)
)
print(f"\nsimulated pair : {match.label()}")
print(f"simulated split: {match.separation_mm:.6f} mm")
print(f"agreement      : "
      f"{abs(abs(analytic_split) - match.separation_mm):.3e} mm")

# The same quantity for titanium, for contrast.
a_beta_ti, c_alpha_ti = 3.3065, 4.6855
split_ti = abs((np.sqrt(2.0) / a_beta_ti - 2.0 / c_alpha_ti) * CAMERA_CONSTANT)
print(f"\ntitanium  split: {split_ti:.6f} mm")
print(f"zirconium split: {abs(analytic_split):.6f} mm")
print(f"ratio          : {abs(analytic_split) / split_ti:.2f} x wider in Zr")

### 8.4 Down $[001]_\beta$: the awkward zone

For contrast, the $[001]_\beta$ composite. No variant is on a rational
zone, so every child pattern is an off-axis section of hcp reciprocal
space. The parent pattern remains the clean four-fold bcc $[001]$ net,
which makes the mismatch visually obvious.

In [ ]:
composite_001 = survey[(0, 0, 1)]
deviations_001 = np.array(
    [p.nearest_zone_axis.deviation_deg for p in composite_001.variant_patterns]
)
values, multiplicities = np.unique(np.round(deviations_001, 3),
                                   return_counts=True)
print("child-zone deviations down [001]_beta:")
for value, multiplicity in zip(values, multiplicities):
    print(f"   {value:6.3f} deg  x {multiplicity}")
print(f"\nminimum deviation: {deviations_001.min():.3f} deg "
      f"(no variant is exact)")

fig = render_composite_saed(
    composite_001.select_variants([1, 5]),
    config=CompositeSAEDPlotConfig(
        annotation=SpotAnnotationConfig(coincidence_tolerance_mm=2.0,
                                        max_labels=18),
        title=("Burgers composite SAED, Zr:  $\\beta$ $[001]$ "
               "(no exact child zone)"),
        figsize=(7.6, 6.4),
    ),
)

### 8.5 Down $[111]_\beta$: the prismatic view

Finally the third canonical view, where the *direction* parallelism
rather than the plane parallelism is aligned with the beam.

In [ ]:
composite_111 = survey[(1, 1, 1)]
exact_111 = [p for p in composite_111.variant_patterns
             if p.nearest_zone_axis.deviation_deg < 1e-6]
print(f"exact variants: {[p.variant_index for p in exact_111]}")
print(f"child zones   : {sorted({p.nearest_zone_axis.label() for p in exact_111})}")

fig = render_composite_saed(
    composite_111.select_variants([p.variant_index for p in exact_111[:2]]),
    config=CompositeSAEDPlotConfig(
        parent_style=SpotStyle(marker="o", color="#1b3a6b", filled=False,
                               size_scale=150.0, edge_width=1.4),
        annotation=SpotAnnotationConfig(coincidence_tolerance_mm=2.0,
                                        max_labels=20),
        title=("Burgers composite SAED, Zr:  $\\beta$ $[111]$ "
               "$\\parallel$ $\\langle 11\\bar{2}0 \\rangle_\\alpha$"),
        figsize=(7.6, 6.4),
    ),
)

## 9. Variant pole figures

The diffraction views above are single-crystal, zone-axis
observables. The complementary bulk observable is the pole figure: for
one parent orientation, where do the basal poles of all twelve
variants land on the specimen sphere?

This is the overlay used to read a measured $\alpha$-Zr $(0001)$ pole
figure and decide which variants actually formed — the entry point to
variant-selection analysis in rolled and recrystallised Zr.

In [ ]:
specimen_frame = ReferenceFrame(
    "specimen", FrameDomain.SPECIMEN, ("x", "y", "z"), Handedness.RIGHT
)
parent_orientation = Orientation(
    rotation=Rotation.from_bunge_euler(0.0, 0.0, 0.0),
    crystal_frame=beta_frame,
    specimen_frame=specimen_frame,
    symmetry=beta_zr.symmetry,
    phase=beta_zr,
)

basal_poles = variant_pole_figure(parent_orientation, burgers, child_plane)
print(basal_poles.describe())

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 6.4))
plot_variant_pole_figure(
    basal_poles,
    title="$(0001)_\\alpha$ poles of all 12 Burgers variants\n"
          "cube-oriented $\\beta$-Zr parent",
    ax=ax,
)
fig.tight_layout()

Each of the twelve variants contributes its basal pole; the poles fall
on the $\{110\}_\beta$ positions of the parent, because the basal plane
*is* a $\{110\}$ plane of the parent by definition of the
relationship. Variants sharing a packet share a pole position — the
six-packet structure of §4.2 read off the pole figure directly.

## Summary

Everything below was computed in this notebook and cross-checked
against an independent closed form:

| quantity | value | independent check |
|---|---|---|
| variants | 12 | $\lvert G_\beta \rvert / 2 = 24/2$ |
| packets | 6 of 2 | $\{110\}$ family multiplicity |
| variant pairs | 66 | $\binom{12}{2}$ |
| distinct intervariant angles | 5 | literature table |
| smallest intervariant angle | $10.529^\circ$ | $\arccos(1/3) - 60^\circ$ |
| residual polar rotation | $5.264^\circ$ | $60^\circ - \arccos(1/\sqrt{3})$ |
| principal strains | $+10.75\%$, $+1.83\%$, $-9.57\%$ | closed forms in §6.2 |
| volume change | $+1.99\%$ | product of the three stretches |
| correspondence denominator | 2 | the Burgers shuffle |
| $\{110\}_\beta / (0002)_\alpha$ split | $1.281$ mm | $(\sqrt{2}/a_\beta - 2/c_\alpha) L\lambda$ |

Two structural conclusions are worth carrying away.

**The mechanism and the misorientation spectrum are the same
geometry.** The two bond angles of the distorted hexagon on
$(110)_\beta$ — $\arccos(1/3) = 70.53^\circ$ and
$\arccos(1/\sqrt{3}) = 54.74^\circ$ — are not merely descriptive.
Measured against the regular $60^\circ$ they give the smallest
intervariant misorientation and the residual polar rotation exactly.
Neither depends on the lattice parameters at all; both are constants
of the bcc lattice.

**Zirconium is not titanium with different numbers.** The
$\{110\}_\beta / (0002)_\alpha$ coincidence, effectively exact in Ti,
splits by $1.28$ mm in Zr — about eight times wider — because
$c_\alpha/a_\alpha$ is $2.5\%$ below ideal. A Burgers analysis
transplanted from the Ti literature without recomputing this will
misread zirconium patterns.

## References

### Normative

- {doc}`../../concepts/orientation_relationships`
- {doc}`../../architecture/phase_transformation_foundation`
- {doc}`../../architecture/orientation_relationship_analysis_foundation`

### Informative

- W. G. Burgers, *On the process of transition of the cubic-body-centred
  modification into the hexagonal-close-packed modification of
  zirconium*, Physica **1** (1934) 561-586. The original paper, on this
  very system.
- N. Gey and M. Humbert, *Characterization of the variant selection
  occurring during the alpha-beta-alpha phase transformations of a cold
  rolled titanium sheet*, Acta Materialia **50** (2002) 277-287.
- S. C. Wang, M. Aindow and M. J. Starink, *Effect of self-accommodation
  on alpha/alpha boundary populations in pure titanium*, Acta Materialia
  **51** (2003) 2485-2503.
- {doc}`18_orientation_relationships_fundamentals` for the KS treatment
  this notebook parallels, and
  {doc}`21_composite_or_diffraction_patterns` for the composite
  diffraction engine in general.